# 🛠️ Oficina Prática: Construindo Tabelas Fato do Zero
### Aprendendo Engenharia de Dados e Pandas na Prática Laboratorial
**Autora:** Luanna Oliveira  
**Objetivo:** Aprender passo a passo como projetar, codificar e gerar uma **Tabela Fato** para o Painel de Qualidade.

---


## 🧠 Conceito 1: O que é a Granularidade de uma Tabela Fato?

Em Sistemas de Informação e Data Warehouse, a primeira pergunta que você deve responder antes de escrever qualquer código é:
> **"O que representa uma linha desta tabela?"**

* Na nossa **`fato_amostras`**, o grão é: **Um tubo/frasco de amostra recebido pelo laboratório**.
* Se um paciente faz EAS e Urocultura, ele terá **duas linhas**, pois são duas amostras/processos distintos.

---


## 🔬 Passo 1: Construindo as primeiras 3 amostras "na mão"
Antes de automatizar, vamos ver como criamos linhas manualmente usando dicionários em Python.

Execute a célula abaixo para ver como o Pandas monta a tabela básica:

In [ ]:
import pandas as pd

# Criamos uma lista de dicionários. Cada dicionário representa 1 tubo/frasco:
minhas_primeiras_amostras = [
    {
        "id_amostra": "AMO-001",
        "exame": "Urina EAS",
        "posto_coleta": "Posto Central",
        "tat_alvo_horas": 4,
        "tat_real_horas": 3.2,
        "status": "Conforme"
    },
    {
        "id_amostra": "AMO-002",
        "exame": "Urocultura",
        "posto_coleta": "Pronto Atendimento",
        "tat_alvo_horas": 48,
        "tat_real_horas": 56.0,
        "status": "Atraso no Laudo (TAT)"
    },
    {
        "id_amostra": "AMO-003",
        "exame": "Urina EAS",
        "posto_coleta": "Posto Central",
        "tat_alvo_horas": 4,
        "tat_real_horas": 2.0,
        "status": "Recoleta Solicitada"
    }
]

# Convertemos para DataFrame
df_teste = pd.DataFrame(minhas_primeiras_amostras)
df_teste


## ⏱️ Passo 2: Trabalhando com Datas e Horas no Python (`datetime`)
Em saúde e laboratório, o tempo é crítico (horas entre coleta, recebimento e liberação).

No Python, usamos a biblioteca nativa `datetime` e `timedelta` (que serve para somar ou subtrair horas/dias).

Execute o exemplo abaixo:

In [ ]:
from datetime import datetime, timedelta

# Vamos fixar uma data de coleta:
data_coleta = datetime(2026, 11, 5, 8, 30) # 05/11/2026 às 08:30
print(f"Horário da Coleta: {data_coleta}")

# A amostra levou 45 minutos no motoboy até chegar na bancada:
data_recebimento = data_coleta + timedelta(minutes=45)
print(f"Horário de Recebimento no Laboratório: {data_recebimento}")

# O exame demorou 3 horas e meia para ser analisado e liberado:
tempo_gasto_horas = 3.5
data_liberacao = data_recebimento + timedelta(hours=tempo_gasto_horas)
print(f"Horário de Liberação do Laudo: {data_liberacao}")


## 🎲 Passo 3: Criando a Lógica de Desvios (Decidindo quem atrasa ou recoleta)
Em um laboratório real, a maioria das amostras passa sem problemas (90-95%), e uma fração sofre desvios.

Como simulamos isso em Sistemas de Informação?
Usamos a função `random.random()`, que sorteia um número decimal entre `0.0` e `1.0` (como se fosse uma porcentagem de 0% a 100%):

* Se o sorteio for menor que 0.03 (3%), definimos como **Recoleta Solicitada**.
* Se for entre 0.03 e 0.06 (3% a 6%), definimos como **Atraso de TAT**.
* O restante (acima de 0.06) é **Conforme** (sucesso normal da rotina).

Teste o sorteio na célula abaixo:

In [ ]:
import random

# Teste simulando 10 amostras para ver o sorteador agir:
for i in range(1, 11):
    sorteio = random.random() # Gera um número aleatório entre 0 e 1
    
    if sorteio < 0.20: # 20% de chance para vermos acontecer no teste
        status = "Recoleta Solicitada (Tubo Quebrado / Não Vedado)"
    elif sorteio < 0.40: # 20% de chance
        status = "Atraso de TAT (Repique / Demora)"
    else:
        status = "Conforme (Liberado no Prazo)"
        
    print(f"Amostra {i:02d} -> Sorteio: {sorteio:.2f} -> Resultado: {status}")


## 🚀 Passo 4: O Seu Primeiro Gerador Completo da `fato_amostras`
Agora vamos juntar tudo em um loop `for` que gera **50 amostras** e cria a tabela oficial!

Leia os comentários no código e execute:

In [ ]:
import pandas as pd
import random
from datetime import datetime, timedelta

# Lista vazia que vai guardar todas as nossas amostras geradas:
tabela_fato_amostras = []

# Configurações do laboratório:
lista_exames = ["Urina EAS", "Urocultura", "Parasitológico de Fezes", "Micologia"]
lista_postos = ["Posto Central", "Posto Shopping", "Pronto Atendimento", "UTI"]

data_base = datetime(2026, 11, 1, 7, 0) # 01/11/2026 às 07:00 da manhã

for numero in range(1, 51): # Vamos gerar 50 amostras
    # 1. Identificador único da amostra (Chave Primária):
    id_amostra = f"AMO-{numero:04d}"
    
    # 2. Escolher aleatoriamente o exame e o posto:
    exame = random.choice(lista_exames)
    posto = random.choice(lista_postos)
    
    # 3. Gerar horário de coleta e recebimento:
    data_coleta = data_base + timedelta(hours=random.randint(1, 48), minutes=random.randint(0, 59))
    data_recebimento = data_coleta + timedelta(minutes=random.randint(30, 90))
    
    # 4. Definir meta de TAT de acordo com o exame:
    if exame == "Urina EAS":
        tat_alvo = 4
    elif exame == "Urocultura":
        tat_alvo = 48
    elif exame == "Parasitológico de Fezes":
        tat_alvo = 24
    else: # Micologia
        tat_alvo = 360 # 15 dias
        
    # 5. Definir a qualidade (Conforme, Recoleta ou Atraso):
    sorteio = random.random()
    if sorteio < 0.05: # 5% de recoleta
        status = "Recoleta Solicitada"
        tat_real = 2.0
        dentro_prazo = "Não Aplicável"
    elif sorteio < 0.12: # 7% de atraso
        status = "Atraso no Laudo (TAT)"
        tat_real = round(tat_alvo * random.uniform(1.2, 1.8), 1)
        dentro_prazo = "Não"
    else: # Conforme
        status = "Conforme"
        tat_real = round(tat_alvo * random.uniform(0.5, 0.9), 1)
        dentro_prazo = "Sim"
        
    # 6. Adicionar a linha na lista:
    tabela_fato_amostras.append({
        "id_amostra": id_amostra,
        "exame": exame,
        "posto_coleta": posto,
        "data_coleta": data_coleta.strftime("%d/%m/%Y %H:%M"),
        "data_recebimento": data_recebimento.strftime("%d/%m/%Y %H:%M"),
        "tat_alvo_horas": tat_alvo,
        "tat_real_horas": tat_real,
        "status_amostra": status,
        "dentro_do_prazo": dentro_prazo
    })

# Converter para DataFrame final:
df_minha_fato = pd.DataFrame(tabela_fato_amostras)
print(f"🎉 Parabéns! Você gerou {len(df_minha_fato)} linhas da sua Tabela Fato!")
display(df_minha_fato.head(10)) # Mostra as 10 primeiras linhas


## 📊 Passo 5: Analisando as métricas da sua Tabela Fato com Pandas
Veja como é fácil responder perguntas de gestão da qualidade com uma linha de código em Pandas:

In [ ]:
print("--- Quantidade de amostras por Status ---")
print(df_minha_fato["status_amostra"].value_counts())

print("\n--- Percentual (%) de cada Status no Laboratório ---")
print((df_minha_fato["status_amostra"].value_counts(normalize=True) * 100).round(1).astype(str) + "%")

print("\n--- Volume de Amostras por Posto de Coleta ---")
print(df_minha_fato["posto_coleta"].value_counts())


## 🚨 Passo 6: Construindo a segunda tabela: `fato_ocorrencias`
Agora vem o segredo da Engenharia de Dados: como criamos a tabela de desvios conectada com a tabela de amostras?

Nós filtramos **apenas as amostras que NÃO foram conformes** e criamos um registro de Não Conformidade com causa e SLA de investigação!

Execute a célula abaixo para ver essa mágica:

In [ ]:
# 1. Filtramos apenas as amostras que tiveram algum problema:
amostras_com_problema = df_minha_fato[df_minha_fato["status_amostra"] != "Conforme"]
print(f"Total de amostras que precisam de investigação: {len(amostras_com_problema)}")

# 2. Criamos a lista de ocorrências:
tabela_fato_ocorrencias = []
motivos_possiveis = [
    "Tubo Rachado na Centrifugação", 
    "Amostra Não Vedada / Vazamento", 
    "Cadastro de Exame Errado", 
    "Necessidade de Repique (Cultura Mista)"
]

contador_nc = 1
for indice, linha in amostras_com_problema.iterrows():
    tabela_fato_ocorrencias.append({
        "id_ocorrencia": f"NC-{contador_nc:03d}",
        "id_amostra": linha["id_amostra"], # Conecta com a fato_amostras!
        "tipo_desvio": linha["status_amostra"],
        "motivo_investigado": random.choice(motivos_possiveis),
        "sla_dias": 2,
        "dias_para_resolver": round(random.uniform(0.5, 3.5), 1),
        "responsavel": "Luanna Oliveira (Biomédica)"
    })
    contador_nc += 1

df_minha_fato_ocorrencias = pd.DataFrame(tabela_fato_ocorrencias)
df_minha_fato_ocorrencias


## 💾 Passo 7: Como salvar a sua tabela em arquivo `.csv`
Para levar seus dados para o Power BI ou Excel, basta usar o método `.to_csv()`:

In [ ]:
# Salva na pasta do seu projeto:
df_minha_fato.to_csv("../data/processed/minha_fato_amostras_teste.csv", index=False, encoding="utf-8-sig")
df_minha_fato_ocorrencias.to_csv("../data/processed/minha_fato_ocorrencias_teste.csv", index=False, encoding="utf-8-sig")
print("✓ Tabelas salvas com sucesso em data/processed/!")
